# Example 3: Stokes flow in a box

## Problem setup

This example builds on Examples 1 and 2 by replacing the prescribed velocity field with one computed numerically. We solve a two-stage problem on the same mesh.

1. **Stokes flow.** Solve the steady Stokes equations on the domain to obtain a divergence-free velocity field $\mathbf{w}$.
2. **Advection-diffusion.** Solve the advection-diffusion equation using $\mathbf{w}$ as the transport velocity, with a uniform volumetric source $f = 1$.

The domain $\Omega$ is a 1 × 1 box with a 0.2 × 0.2 inlet channel protruding from the top-left and a 0.2 × 0.2 outlet channel protruding from the bottom-right (both extending in the $x$-direction). Fluid enters through the inlet and exits through the outlet, so the flow path crosses the box diagonally.

:::{image} domain.jpg
:align: center
:width: 75%
:::

## The Stokes equations

The Stokes equations describe viscous flow at very low Reynolds number, where inertia is negligible compared to viscosity. The strong form is

$$
-\mu \Delta \mathbf{u} + \nabla p = \mathbf{0}, \qquad \nabla \cdot \mathbf{u} = 0 \quad \text{in } \Omega,
$$

where $\mathbf{u}$ is the velocity, $p$ is the pressure, and $\mu$ is the dynamic viscosity.

**Discretisation.** We use Taylor–Hood elements: a continuous piecewise-quadratic space (P2) for velocity and a continuous piecewise-linear space (P1) for pressure. This pairing satisfies the inf-sup (LBB) condition, which prevents spurious pressure oscillations.

The weak form is: find $(\mathbf{u}, p) \in V_h \times Q_h$ such that for all test functions $(\mathbf{v}, q)$,

$$
\mu\int_\Omega \nabla\mathbf{u} : \nabla\mathbf{v}\,\mathrm{d}x
- \int_\Omega p\,\nabla\cdot\mathbf{v}\,\mathrm{d}x
- \int_\Omega (\nabla\cdot\mathbf{u})\,q\,\mathrm{d}x = 0.
$$

The pressure and incompressibility terms have opposite signs, giving the symmetric saddle-point structure characteristic of the Stokes problem.

## Boundary conditions

### Stokes

| Boundary | Location | Condition |
|---|---|---|
| Inlet | Left face of inlet channel | Plug flow: $\mathbf{u} = (0.1,\, 0)$ m s$^{-1}$ (Dirichlet) |
| Outlet | Right face of outlet channel | Natural — zero normal stress |
| Walls | All remaining faces | No-slip: $\mathbf{u} = \mathbf{0}$ (Dirichlet) |

The outlet condition is *natural*: no term is added to the weak form for that boundary, so the variational principle automatically enforces zero normal stress there.

### Advection-diffusion

| Boundary | $\mathbf{w}\cdot\mathbf{n}$ | Advection | Diffusion |
|---|---|---|---|
| Inlet | $< 0$ (inflow) | Prescribed inflow: $u = 0$ | Nitsche: $u = 0$ |
| Outlet | $> 0$ (outflow) | Natural upwind outflow | Natural (zero normal flux) |
| Walls | $= 0$ | No term | Natural (zero normal flux) |

The advection-diffusion boundary treatment follows the same pattern as Examples 1 and 2.

## Advection-diffusion formulation

The weak form — upwind advection at interior faces, SIPG diffusion, and a Nitsche condition at the inlet — is identical to Examples 1 and 2. The only change here is that the velocity field $\mathbf{w}$ comes from the Stokes solve rather than being prescribed analytically.

We take $D = 10^{-5}$ and a uniform volumetric source $f = 1$, giving a large Péclet number so the solution is strongly advection-dominated.

In [10]:
from pathlib import Path

from mpi4py import MPI
from petsc4py import PETSc

import numpy as np
import ufl
import basix.ufl
import dolfinx
from dolfinx import fem
from dolfinx.fem.petsc import LinearProblem, NonlinearProblem
from dolfinx.io import XDMFFile, VTXWriter
import io4dolfinx

COMM = MPI.COMM_WORLD
CHECKPOINT = Path("sim_checkpoint.bp")

INLET_ID = 7
OUTLET_ID = 8
WALLS_ID = 9

## Loading the mesh

The mesh and facet tags are read from XDMF files produced by the external mesher. The facet tags identify which boundary each face belongs to; they are needed to apply boundary conditions and to compute boundary fluxes later.

In [11]:
with XDMFFile(COMM, "mesh_data/mesh_domains.xdmf", "r") as f:
    msh = f.read_mesh(name="Grid")
msh.topology.create_connectivity(msh.topology.dim, msh.topology.dim - 1)
msh.topology.create_connectivity(msh.topology.dim - 1, msh.topology.dim)
with XDMFFile(COMM, "mesh_data/mesh_boundaries.xdmf", "r") as f:
    facet_mt = f.read_meshtags(msh, name="Grid")

## Solving Stokes flow

We build a Taylor–Hood mixed function space on the mesh and split trial and test functions into velocity and pressure components. Dirichlet conditions are applied at the inlet (plug flow) and all wall faces (no-slip); the outlet is left free so the variational principle enforces zero normal stress there.

The system has a symmetric saddle-point structure and is solved with a direct LU factorisation via MUMPS. After solving, the velocity sub-solution is extracted into a standalone CG2 vector field for use in the advection-diffusion step.

In [12]:
gdim = msh.geometry.dim
fdim = msh.topology.dim - 1

P2 = basix.ufl.element("Lagrange", msh.topology.cell_name(), 2, shape=(gdim,))
P1 = basix.ufl.element("Lagrange", msh.topology.cell_name(), 1)
W = fem.functionspace(msh, basix.ufl.mixed_element([P2, P1]))
V_sub, _ = W.sub(0).collapse()

u_in = fem.Function(V_sub)
u_in.interpolate(
    lambda x: np.vstack([0.1 * np.ones(x.shape[1]), np.zeros(x.shape[1])])
)
bcs = [
    fem.dirichletbc(
        u_in,
        fem.locate_dofs_topological(
            (W.sub(0), V_sub), fdim, facet_mt.find(INLET_ID)
        ),
        W.sub(0),
    ),
    fem.dirichletbc(
        fem.Function(V_sub),
        fem.locate_dofs_topological(
            (W.sub(0), V_sub), fdim, facet_mt.find(WALLS_ID)
        ),
        W.sub(0),
    ),
]

u_tr, p_tr = ufl.TrialFunctions(W)
v_s, q = ufl.TestFunctions(W)
mu = fem.Constant(msh, PETSc.ScalarType(1.0))
a = (
    mu * ufl.inner(ufl.grad(u_tr), ufl.grad(v_s)) * ufl.dx
    - ufl.inner(p_tr, ufl.div(v_s)) * ufl.dx
    - ufl.inner(ufl.div(u_tr), q) * ufl.dx
)
L = ufl.inner(fem.Constant(msh, PETSc.ScalarType((0.0,) * gdim)), v_s) * ufl.dx

stokes_problem = LinearProblem(
    a,
    L,
    bcs=bcs,
    petsc_options_prefix="stokes",
    petsc_options={
        "ksp_type": "preonly",
        "pc_type": "lu",
        "pc_factor_mat_solver_type": "mumps",
        "ksp_error_if_not_converged": True,
    },
)
wh = stokes_problem.solve()

el_cg2 = basix.ufl.element("Lagrange", msh.topology.cell_name(), 2, shape=(gdim,))
V_cg2 = fem.functionspace(msh, el_cg2)
u_h = fem.Function(V_cg2, name="velocity")
u_h.interpolate(wh.sub(0).collapse())
u_h.x.scatter_forward()

## Saving to checkpoint

We write the mesh, facet tags, and velocity field to an `io4dolfinx` checkpoint. This separates the Stokes solve from the advection-diffusion solve, so the transport problem can be re-run with different parameters — for example a different diffusivity — without repeating the Stokes computation.

In [13]:
io4dolfinx.write_mesh(CHECKPOINT, msh)
io4dolfinx.write_meshtags(CHECKPOINT, msh, facet_mt, meshtag_name="facets")
io4dolfinx.write_function(CHECKPOINT, u_h, time=0.0, name="velocity")

## Loading the velocity field

We reload the mesh, facet tags, and Stokes velocity from the checkpoint. From this point the advection-diffusion solve is entirely independent of the Stokes solve — it only requires the velocity field stored in the checkpoint.

In [14]:
msh = io4dolfinx.read_mesh(CHECKPOINT, COMM)
msh.topology.create_connectivity(msh.topology.dim, msh.topology.dim - 1)
msh.topology.create_connectivity(msh.topology.dim - 1, msh.topology.dim)
facet_mt = io4dolfinx.read_meshtags(CHECKPOINT, msh, meshtag_name="facets")

gdim = msh.geometry.dim
el_cg2 = basix.ufl.element("Lagrange", msh.topology.cell_name(), 2, shape=(gdim,))
V_cg2 = fem.functionspace(msh, el_cg2)
w = fem.Function(V_cg2, name="velocity")
io4dolfinx.read_function(CHECKPOINT, w, time=0.0, name="velocity")

## Setting up the advection-diffusion problem

We use a DG1 function space. The upwind indicator $\lambda$ classifies each face as outflow ($\lambda = 1$) or inflow ($\lambda = 0$) based on the sign of $\mathbf{w}\cdot\mathbf{n}$, following the same pattern as Examples 1 and 2.

In [15]:
V = fem.functionspace(msh, ("DG", 1))
u = fem.Function(V)
v = ufl.TestFunction(V)

n = ufl.FacetNormal(msh)
h = ufl.CellDiameter(msh)
ds = ufl.Measure("ds", domain=msh, subdomain_data=facet_mt)
dS, dx = ufl.dS, ufl.dx

D = fem.Constant(msh, PETSc.ScalarType(1e-5))
penalty = fem.Constant(msh, PETSc.ScalarType(10))
u_inlet = fem.Constant(msh, PETSc.ScalarType(0.0))
f_source = fem.Constant(msh, PETSc.ScalarType(1.0))

lmbda = ufl.conditional(ufl.gt(ufl.dot(w, n), 0), 1, 0)

## Assembling the weak form

The residual $F$ is assembled in the same parts as in Examples 1 and 2: advection bulk and interface terms, SIPG diffusion, the Nitsche inlet condition, and the volumetric source. The named variables `F_inlet_adv`, `F_outlet_surf`, and `F_inlet_nitsche` are kept so that the flux verification step can isolate individual boundary contributions.

In [16]:
# Advection: bulk and interior upwind flux
F = -ufl.inner(w * u, ufl.grad(v)) * dx
F += ufl.inner(2 * ufl.avg(lmbda * w * u), ufl.jump(v, n)) * dS

# Advection: boundary terms (kept named for flux verification)
F_inlet_adv = -ufl.inner((1 - lmbda) * ufl.dot(w, n) * u_inlet, v) * ds(INLET_ID)
F_outlet_surf = ufl.inner(lmbda * ufl.dot(w, n) * u, v) * ds(OUTLET_ID)
F += F_inlet_adv + F_outlet_surf

# Diffusion (SIPG): bulk and interior interface terms
F += D * ufl.inner(ufl.grad(u), ufl.grad(v)) * dx
F += -D * ufl.inner(ufl.avg(ufl.grad(u)), ufl.jump(v, n)) * dS
F += -D * ufl.inner(ufl.jump(u, n), ufl.avg(ufl.grad(v))) * dS
F += D * (penalty / ufl.avg(h)) * ufl.inner(ufl.jump(u, n), ufl.jump(v, n)) * dS

# Nitsche inlet condition (weak Dirichlet)
F_inlet_nitsche = D * (
    -ufl.inner(ufl.grad(u), v * n) * ds(INLET_ID)
    - ufl.inner(ufl.grad(v), (u - u_inlet) * n) * ds(INLET_ID)
    + (penalty / h) * ufl.inner(u - u_inlet, v) * ds(INLET_ID)
)
F += F_inlet_nitsche
F_inlet_terms = F_inlet_adv + F_inlet_nitsche

# Volumetric source
F += -ufl.inner(f_source, v) * dx

## Solving and writing output

Although the advection-diffusion equation is linear in $u$, it is posed as a nonlinear residual and solved with SNES. This is equivalent to a single Newton step from the zero initial guess, and keeps the formulation consistent with any future nonlinear extensions. The solution is written to a VTX file for visualisation in ParaView.

In [17]:
adv_diff_problem = NonlinearProblem(
    F,
    u,
    J=ufl.derivative(F, u),
    petsc_options_prefix="adv_diff",
    petsc_options={
        "snes_atol": 1e-12,
        "snes_rtol": 1e-12,
        "snes_max_it": 30,
        "ksp_type": "preonly",
        "pc_type": "lu",
        "pc_factor_mat_solver_type": "mumps",
    },
)
u = adv_diff_problem.solve()
u.x.scatter_forward()

writer = VTXWriter(COMM, "mwe_box.bp", u, "BP5")
writer.write(t=0.0)

## Flux balance verification

We use the consistent-flux technique to verify global mass conservation. For each boundary, we assemble the full residual vector and sum the entries belonging to cells adjacent to that boundary. If the method is conservative, the total boundary flux should equal the volumetric source integral to machine precision.

Expected results:
- **Outlet flux** $\approx$ source integral — most tracer leaves through the outlet.
- **Inlet flux** $\approx 0$ — the inlet concentration is zero, so no tracer enters by diffusion.
- **Wall flux** $\approx 0$ — the natural BC gives zero normal diffusive flux, and $\mathbf{w}\cdot\mathbf{n} = 0$ at no-slip walls removes advective transport.

In [18]:
tdim = msh.topology.dim
fdim = tdim - 1
msh.topology.create_connectivity(fdim, tdim)
msh.topology.create_connectivity(tdim, tdim)
owned_size = V.dofmap.index_map.size_local * V.dofmap.index_map_bs


def get_owned_dofs(marker):
    facets = facet_mt.find(marker)
    f_to_c = msh.topology.connectivity(fdim, tdim)
    cells = np.unique(np.concatenate([f_to_c.links(f) for f in facets]))
    dofs = fem.locate_dofs_topological(V, tdim, cells)
    return dofs[dofs < owned_size]


def compute_consistent_flux(residual_form, dofs):
    residual = fem.assemble_vector(residual_form)
    residual.scatter_reverse(dolfinx.la.InsertMode.add)
    residual.scatter_forward()
    local_flux = np.sum(residual.array[dofs])
    return msh.comm.allreduce(local_flux, op=MPI.SUM)


inlet_dofs = get_owned_dofs(INLET_ID)
outlet_dofs = get_owned_dofs(OUTLET_ID)
wall_dofs = get_owned_dofs(WALLS_ID)

F_no_outlet = fem.form(F - F_outlet_surf)
F_no_inlet = fem.form(F - F_inlet_terms)
F_form = fem.form(F)

flux_outlet = -compute_consistent_flux(F_no_outlet, outlet_dofs)
flux_inlet = -compute_consistent_flux(F_no_inlet, inlet_dofs)
flux_wall = compute_consistent_flux(F_form, wall_dofs)

source_total = msh.comm.allreduce(
    fem.assemble_scalar(fem.form(f_source * dx)), op=MPI.SUM
)

consist_total = flux_outlet + flux_wall + flux_inlet


def pct(val):
    return 100 * val / source_total


print(f"Source integral : {source_total:.6e}")
print()
print(f"{'Flux outlet':20s} {flux_outlet:14.6e}")
print(f"{'Flux inlet':20s} {flux_inlet:14.6e}")
print(f"{'Flux wall':20s} {flux_wall:14.6e}")
print()
c_bal = source_total - consist_total
print(f"{'Total flux':20s} {consist_total:14.6e}")
print(f"{'Balance residual':20s} {c_bal:14.6e} ({pct(c_bal):+.2f}%)")

Source integral : 1.080000e+00

Flux outlet            1.079979e+00
Flux inlet             2.121287e-05
Flux wall              1.477355e-16

Total flux             1.080000e+00
Balance residual       4.218847e-15 (+0.00%)
